In [2]:
import os                                   #imports
import tensorflow as tf
import time
import pandas as pd
import gensim.downloader as api
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow import keras
from sklearn.metrics import f1_score
from tensorflow.keras.metrics import Precision,Recall,CategoricalAccuracy
from tensorflow.keras.callbacks import ModelCheckpoint,EarlyStopping
from tensorflow.keras.layers import Dropout,Dense,Embedding,LSTM,TextVectorization,Bidirectional,GRU,Convolution1D,GlobalMaxPooling1D
from tensorflow.keras.models import Sequential

In [3]:
df = pd.read_csv("train.csv") # reading the training dataset

In [4]:
df

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0
...,...,...,...,...,...,...,...,...
159566,ffe987279560d7ff,""":::::And for the second time of asking, when ...",0,0,0,0,0,0
159567,ffea4adeee384e90,You should be ashamed of yourself \n\nThat is ...,0,0,0,0,0,0
159568,ffee36eab5c267c9,"Spitzer \n\nUmm, theres no actual article for ...",0,0,0,0,0,0
159569,fff125370e4aaaf3,And it looks like it was actually you who put ...,0,0,0,0,0,0


In [5]:
df.drop_duplicates(inplace = True)
df.drop("id",axis=1,inplace = True)

In [6]:
X = df["comment_text"]
y = df.iloc[:,1:].values

In [7]:
x_train,x_temp,y_train,y_temp = train_test_split(X,y,test_size = 0.3,random_state = 42)
x_val,x_test,y_val,y_test = train_test_split(x_temp,y_temp,test_size = 0.4,random_state = 42)

In [8]:
y

array([[0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       ...,
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0]])

In [9]:
max_features = 100000 # this is the number of words in the vocab
vectorizer = TextVectorization(max_tokens = max_features,
                               output_sequence_length=200,
                               output_mode = 'int')

In [10]:
vectorizer.adapt(x_train.values) # we are making the vectorizer learn the text

In [11]:
train_vectorized_text = vectorizer(x_train.values) #converting all the comments into vectors
val_vectorized_text = vectorizer(x_val.values)
test_vectorized_text = vectorizer(x_test.values)

In [12]:
train_dataset = tf.data.Dataset.from_tensor_slices((train_vectorized_text,y_train))
val_dataset = tf.data.Dataset.from_tensor_slices((val_vectorized_text,y_val))
test_dataset = tf.data.Dataset.from_tensor_slices((test_vectorized_text,y_test))

train_dataset = train_dataset.cache()
train_dataset = train_dataset.shuffle(len(x_train))
train_dataset = train_dataset.batch(64)
train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)

val_dataset = val_dataset.batch(64)
test_dataset = test_dataset.batch(64)

val_dataset = val_dataset.prefetch(tf.data.AUTOTUNE)
test_dataset = test_dataset.prefetch(tf.data.AUTOTUNE)

In [13]:
glove = api.load("glove-wiki-gigaword-100") #GloVe integration
vocab = vectorizer.get_vocabulary()
embedding_matrix = []
embedding_dim = glove.vector_size
mean = np.mean(glove.vectors)
std = np.std(glove.vectors)
embedding_matrix = np.random.normal(mean,std,(len(vocab),embedding_dim))
for idx,word in enumerate(vocab):
  if word in glove:
    embedding_matrix[idx] = glove[word]

[==================================================] 100.0% 128.1/128.1MB downloaded


In [14]:
hits = 0
for word in vocab:
    if word in glove:
        hits += 1

print("Coverage:", hits / len(vocab))
#as we can see the coverage is only 58% so we must keep the trainable parameter set to True during training

Coverage: 0.58313


In [15]:
label_count = np.sum(y,axis=0)
total_samples = y.shape[0]
pos_weights = total_samples/(2*(label_count))
print("Positive class weights per label:")
for label,weight in zip(df.columns[1:],pos_weights):
  print(label,":",weight)

Positive class weights per label:
toxic : 5.216784359879691
severe_toxic : 50.02225705329153
obscene : 9.443188543022844
threat : 166.9152719665272
insult : 10.128919639456646
identity_hate : 56.78683274021353


In [16]:
def weighted_bce(pos_weights):
  pos_weights_tensor = tf.constant(pos_weights,dtype = tf.float32)
  def loss(y_true,y_pred):
    eps = 1e-7
    y_pred = tf.clip_by_value(y_pred,eps,1-eps) #for stable gradient
    loss = -(pos_weights_tensor * y_true * tf.math.log(y_pred) + (1 - y_true) * tf.math.log(1 - y_pred))
    return tf.reduce_mean(loss) #computes average loss of the 6 classes, this scalar is used for backprop\
  return loss

In [17]:
def build_model(layer = None,cnn = False):
  tf.keras.backend.clear_session() # to clear gpu memory after each training
  if cnn == False:
    model = Sequential([
        Embedding(input_dim = len(vocab),output_dim = embedding_dim,weights = [embedding_matrix],mask_zero = True,trainable = True), #using pretrained vectors for better results
        Bidirectional(layer(64)),
        Dense(128, activation='relu'),
        Dropout(0.2),
        Dense(128, activation='relu'),
        Dense(6, activation='sigmoid')
    ])
  else:
    model = Sequential([
        Embedding(input_dim = len(vocab),output_dim = embedding_dim,weights = [embedding_matrix],trainable = True),
        Convolution1D(kernel_size = 5,activation = 'relu',filters=64),
        Dropout(0.2),
        Convolution1D(kernel_size = 5,activation = 'relu',filters=128),
        GlobalMaxPooling1D(),
        Dense(128, activation='relu'),
        Dropout(0.2),
        Dense(128, activation='relu'),
        Dense(6, activation='sigmoid')
    ])
  model.compile(
      optimizer='adam',
      loss=weighted_bce(pos_weights),
      metrics=[tf.keras.metrics.AUC(name = "auc",multi_label=True),
                tf.keras.metrics.Precision(name = "precision",thresholds=0.3),
                tf.keras.metrics.Recall(name = "recall",thresholds=0.3)]
  )
  return model


In [18]:
model_lstm = build_model(LSTM)
model_gru  = build_model(GRU)
cnn_model = build_model(cnn = True)

In [19]:
callbacks = [
    EarlyStopping(
        monitor="val_auc",
        patience=2,
        mode="max",
        restore_best_weights=True
    ),
    ModelCheckpoint(
        "best_model.keras",
        monitor="val_auc",
        mode="max",
        save_best_only=True
    )
]

In [20]:
start = time.time()
history_gru = model_gru.fit(train_dataset,epochs = 10, validation_data=val_dataset,callbacks = callbacks)
print("Time taken for training by GRU:", time.time() - start)

Epoch 1/10
1746/1746 ━━━━━━━━━━━━━━━━━━━━ 60s 30ms/step - auc: 0.9263 - loss: 0.4379 - precision: 0.1739 - recall: 0.8941 - val_auc: 0.9820 - val_loss: 0.2475 - val_precision: 0.2857 - val_recall: 0.9331
Epoch 2/10
1746/1746 ━━━━━━━━━━━━━━━━━━━━ 51s 29ms/step - auc: 0.9865 - loss: 0.1910 - precision: 0.3541 - recall: 0.9600 - val_auc: 0.9775 - val_loss: 0.2991 - val_precision: 0.4218 - val_recall: 0.9309
Epoch 3/10
1746/1746 ━━━━━━━━━━━━━━━━━━━━ 49s 28ms/step - auc: 0.9912 - loss: 0.1404 - precision: 0.4381 - recall: 0.9759 - val_auc: 0.9782 - val_loss: 0.2687 - val_precision: 0.3764 - val_recall: 0.9271
Time taken for training by GRU: 160.80491161346436


In [21]:
start = time.time()
history_lstm = model_lstm.fit(train_dataset,epochs = 10, validation_data=val_dataset,callbacks = callbacks)
print("Time taken for training by LSTM:", time.time() - start)

Epoch 1/10
1746/1746 ━━━━━━━━━━━━━━━━━━━━ 56s 30ms/step - auc: 0.9288 - loss: 0.4365 - precision: 0.1772 - recall: 0.8905 - val_auc: 0.9796 - val_loss: 0.2443 - val_precision: 0.3009 - val_recall: 0.9420
Epoch 2/10
1746/1746 ━━━━━━━━━━━━━━━━━━━━ 52s 30ms/step - auc: 0.9854 - loss: 0.1951 - precision: 0.3469 - recall: 0.9579 - val_auc: 0.9775 - val_loss: 0.2782 - val_precision: 0.3983 - val_recall: 0.9328
Epoch 3/10
1746/1746 ━━━━━━━━━━━━━━━━━━━━ 51s 29ms/step - auc: 0.9909 - loss: 0.1398 - precision: 0.4450 - recall: 0.9743 - val_auc: 0.9770 - val_loss: 0.2919 - val_precision: 0.3763 - val_recall: 0.9374
Time taken for training by LSTM: 159.30593705177307


In [22]:
start = time.time()
history_cnn = cnn_model.fit(train_dataset,epochs = 10, validation_data=val_dataset,callbacks = callbacks)
print("Time taken for training by CNN:", time.time() - start)

Epoch 1/10
1746/1746 ━━━━━━━━━━━━━━━━━━━━ 32s 11ms/step - auc: 0.9121 - loss: 0.4806 - precision: 0.1640 - recall: 0.8680 - val_auc: 0.9761 - val_loss: 0.2745 - val_precision: 0.2949 - val_recall: 0.9379
Epoch 2/10
1746/1746 ━━━━━━━━━━━━━━━━━━━━ 14s 8ms/step - auc: 0.9772 - loss: 0.2532 - precision: 0.2860 - recall: 0.9443 - val_auc: 0.9783 - val_loss: 0.2588 - val_precision: 0.3037 - val_recall: 0.9334
Epoch 3/10
1746/1746 ━━━━━━━━━━━━━━━━━━━━ 14s 8ms/step - auc: 0.9861 - loss: 0.1872 - precision: 0.3534 - recall: 0.9612 - val_auc: 0.9783 - val_loss: 0.2606 - val_precision: 0.3541 - val_recall: 0.9398
Epoch 4/10
1746/1746 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - auc: 0.9887 - loss: 0.1653 - precision: 0.3868 - recall: 0.9711 - val_auc: 0.9732 - val_loss: 0.3153 - val_precision: 0.3693 - val_recall: 0.9153
Time taken for training by CNN: 73.56560349464417


In [23]:
print(f"Max AUC for GRU: {max(history_gru.history['val_auc'])}")
print(f"Max AUC for LSTM: {max(history_lstm.history['val_auc'])}")
print(f"Max AUC for CNN: {max(history_cnn.history['val_auc'])}")

Max AUC for GRU: 0.9819989204406738
Max AUC for LSTM: 0.9796479344367981
Max AUC for CNN: 0.9783210754394531


In [24]:
print(f"Max Precision for GRU: {max(history_gru.history['val_precision'])}")
print(f"Max Precision for LSTM: {max(history_lstm.history['val_precision'])}")
print(f"Max Precision for CNN: {max(history_cnn.history['val_precision'])}")

Max Precision for GRU: 0.4218063950538635
Max Precision for LSTM: 0.3983045220375061
Max Precision for CNN: 0.3693283796310425


In [25]:
print(f"Max Recall for GRU: {max(history_gru.history['val_recall'])}")
print(f"Max Recall for LSTM: {max(history_lstm.history['val_recall'])}")
print(f"Max Recall for CNN: {max(history_cnn.history['val_recall'])}")

Max Recall for GRU: 0.9331321716308594
Max Recall for LSTM: 0.942026674747467
Max Recall for CNN: 0.9398030638694763


In [26]:

def macro_f1_worker(model_name, test_set=True):
    all_preds = []
    all_true = []
    dataset = test_dataset if test_set else val_dataset

    print(f"Running inference on {'test' if test_set else 'validation'} set...")
    for x_batch, y_batch in dataset:
        preds = model_name(x_batch, training=False).numpy()
        all_preds.append(preds)
        all_true.append(y_batch.numpy())

    all_preds = np.vstack(all_preds)
    all_true = np.vstack(all_true)
    return all_preds, all_true

def macro_f1_results(all_preds, all_true, threshold):
    preds_binary = (all_preds > threshold).astype(int)
    return f1_score(all_true, preds_binary, average='macro')


In [27]:
def varying_thresholds(model_name):
    result_li = []

    # CALL WORKER ONCE: This is the expensive GPU part
    # We use val_dataset (test_set=False) for threshold tuning
    all_preds, all_true = macro_f1_worker(model_name, test_set=False)

    print("Calculating scores for various thresholds...")
    for i in range(1, 11, 1):
        threshold = i/10
        # This part is now just CPU math and takes milliseconds
        score = macro_f1_results(all_preds, all_true, threshold)
        result_li.append([threshold, score])

    return result_li

In [28]:
results_gru = varying_thresholds(model_gru)
best_threshold_pair_gru = max(results_gru,key = lambda x: x[1])

results_lstm = varying_thresholds(model_lstm)
best_threshold_pair_lstm = max(results_lstm,key = lambda x: x[1])

results_cnn = varying_thresholds(cnn_model)
best_threshold_pair_cnn = max(results_cnn,key = lambda x: x[1])

Running inference on validation set...
Calculating scores for various thresholds...
Running inference on validation set...
Calculating scores for various thresholds...
Running inference on validation set...
Calculating scores for various thresholds...


In [29]:
print("Best Threshold for GRU is: ",best_threshold_pair_gru[0])
print("Best Threshold for LSTM is: ",best_threshold_pair_lstm[0])
print("Best Threshold for CNN is: ",best_threshold_pair_cnn[0])

Best Threshold for GRU is:  0.9
Best Threshold for LSTM is:  0.9
Best Threshold for CNN is:  0.8


In [30]:
def score_comment(comment,model_name,best_threshold):
  flagged = False
  input_str = vectorizer([comment])
  res = model_name(input_str,training = False).numpy()
  text = ' '
  for idx,cols in enumerate(df.columns[1:]):
    is_toxic = res[0][idx] > best_threshold
    text += '{}: {}  '.format(cols,is_toxic)
    if is_toxic:
      flagged = True
  if flagged == False:
    print("\n\n\n Safe")
  return text

In [31]:
def predictor(ip_text,model_name,best_threshold):
  pred = model_name(vectorizer([ip_text]),training = False).numpy()
  labels = ["toxic",'severe_toxic',"obscene","threat","insult","identity_hate"]
  threshold = best_threshold
  pred_labels = (pred > threshold).astype(int)
  flagged = False
  print("This comment is: ")
  for i in range(len(labels)):
    if pred_labels[0][i] == 1:
      print(labels[i])
      flagged = True
  if flagged == False:
    print("safe")